# Fill missing fund names from GLEIF (run on a machine with internet; updates hf_Valeri.xlsx)

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests

DATA = Path.cwd() / "build" / "Data"
if not DATA.exists():
    DATA = Path.cwd().parent / "build" / "Data"
REPO = DATA.parents[1]
funds = pd.read_excel(DATA / "hf_Valeri.xlsx")
funds["entity_id"] = funds["entity_id"].str.strip().str.upper()
print("missing names:", funds["name"].isna().sum())

In [ ]:
def gleif_name(lei):
    r = requests.get(f"https://api.gleif.org/api/v1/lei-records/{lei}", timeout=30)
    time.sleep(1.05)
    if not r.ok:
        return None
    ent = r.json().get("data", {}).get("attributes", {}).get("entity", {})
    return (ent.get("legalName") or {}).get("name")

missing = funds["name"].isna()
funds.loc[missing, "name"] = funds.loc[missing, "entity_id"].map(gleif_name)
funds.to_excel(DATA / "hf_Valeri.xlsx", index=False)
print("still missing:", funds["name"].isna().sum())